# Tutorial 3: Schnorr Signatures by Hand


**Audience:** Completed Tutorials 1-2

**Prerequisites:** Scalar and Point operations

**Learning goals:**
- Construct a BIP340 Schnorr signature step by step
- Verify the signature using the Schnorr equation
- Understand BIP340 conventions (even-y, tagged hashes)


## What this notebook covers

1. Key generation with BIP340 even-y normalization
2. Nonce generation and commitment
3. Challenge hash computation (BIP340 tagged hash)
4. Signature computation: s = k + e·x
5. Verification: s·G == R + e·P
6. Exercise: nonce reuse attack


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash


## Generate a Keypair

A Schnorr keypair is a random scalar x (private key) and the point P = x·G
(public key). BIP340 requires P to have an even y-coordinate. If it doesn't,
we negate both x and P.


In [ ]:
x = Scalar.random()  # private key
P = int(x) * G       # public key

# BIP340: negate if P has odd y
if not P.has_even_y():
    x = -x
    P = -P

print(f"Private key: {int(x)}")
print(f"Public key (even y): ({P.x}, {P.y})")


## Choose a Message

The message is an arbitrary byte string. In Bitcoin, this would be
the transaction hash.


In [ ]:
message = b"Hello, FROST!"
print(f"Message: {message}")
print(f"Message hex: {message.hex()}")


## Generate a Nonce

The signer picks a random nonce k and computes the nonce commitment R = k·G.
Like the public key, R must have even y for BIP340.


In [ ]:
k = Scalar.random()
R = int(k) * G

# BIP340: negate k if R has odd y
if not R.has_even_y():
    k = -k
    R = -R

print(f"Nonce commitment R: ({R.x}, ...)")
print(f"R has even y? {R.has_even_y()}")


## Compute the Challenge

The challenge e is a tagged hash: H("BIP0340/challenge", R_x || P_x || m).
Tagged hashes use domain separation to prevent cross-protocol attacks.


In [ ]:
challenge_bytes = tagged_hash(
    "BIP0340/challenge",
    R.to_bytes_xonly() + P.to_bytes_xonly() + message,
)
e = int.from_bytes(challenge_bytes, "big") % Q
print(f"Challenge e = {e}")


## Compute the Signature

The signature scalar is s = k + e·x. The full BIP340 signature is
the 64-byte concatenation (R_x, s).


In [ ]:
s = k + Scalar(e) * x
print(f"Signature scalar s = {int(s)}")

# The BIP340 signature is (R_x, s) as 64 bytes
sig = R.to_bytes_xonly() + int(s).to_bytes(32, "big")
print(f"Signature ({len(sig)} bytes): {sig.hex()[:40]}...")


## Verify

Verification checks the equation s·G == R + e·P.


In [ ]:
lhs = int(s) * G
rhs = R + (e * P)
print(f"s·G == R + e·P? {lhs == rhs}")


## Why Verification Works

The algebra is straightforward:

    s·G = (k + e·x)·G = k·G + e·x·G = R + e·P

The verifier never learns k or x, only R, P, and s. The hash e binds
the signature to the specific message and public key.


## Exercise: Nonce Reuse Attack

Sign two different messages with the same nonce k. Show that the private
key can be recovered. This is why nonces must NEVER be reused.


In [ ]:
# Two messages, same nonce
m1 = b"message one"
m2 = b"message two"

e1_bytes = tagged_hash("BIP0340/challenge", R.to_bytes_xonly() + P.to_bytes_xonly() + m1)
e1 = int.from_bytes(e1_bytes, "big") % Q

e2_bytes = tagged_hash("BIP0340/challenge", R.to_bytes_xonly() + P.to_bytes_xonly() + m2)
e2 = int.from_bytes(e2_bytes, "big") % Q

s1 = k + Scalar(e1) * x
s2 = k + Scalar(e2) * x

# Recover private key: x = (s1 - s2) / (e1 - e2)
recovered = (s1 - s2) * Scalar(e1 - e2).inv()
print(f"Original private key:  {int(x)}")
print(f"Recovered private key: {int(recovered)}")
print(f"Match? {int(x) == int(recovered)}")


## Extension

This is exactly why FROST uses *binding values*. If nonces are simply
added in a threshold scheme, a malicious participant can manipulate the
group nonce to create a nonce-reuse situation. Binding values prevent
this by tying each participant's nonce to the full set of commitments.
